# 03 · Evaluation & model selection

Produced by `python -m ml.src.evaluate --experiment <id>` (temperature and confidence threshold fitted on **validation**; test evaluated once) and `python -m ml.src.select_model`.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'ml' / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
DATA_DIR = os.environ.get('WASTEWISE_DATA_DIR', str(ROOT / 'ml/datasets/processed'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 160)

In [ ]:
from IPython.display import Image as Img, Markdown
display(Markdown(Path('ml/reports/model_comparison.md').read_text()))

In [ ]:
from ml.src.taxonomy import CATEGORIES
evals = {p.parent.name: json.load(open(p)) for p in sorted(Path('ml/reports').glob('*/evaluation.json'))}
for k, e in evals.items():
    print(k); display(pd.DataFrame(e['test']['per_class']).T.loc[list(CATEGORIES)].round(3))

In [ ]:
sel = json.load(open('ml/reports/model_comparison.json'))['selected']; print('selected:', sel)
Img(f'ml/reports/{sel}/confusion_matrix.png')

In [ ]:
Img(f'ml/reports/{sel}/selective_curve.png')

In [ ]:
e = evals[sel]
pd.Series({'temperature': e['temperature'], 'threshold': e['confidence_threshold'], **e['test_selective'],
           'ood_rejection_rate': e['ood']['rejection_rate']})

### Safety-relevant classes and most frequent confusions

In [ ]:
display(pd.DataFrame(e['safety_classes']).T)
pd.DataFrame(e['top_confusions'])

### Misclassified test images (safety-relevant errors first)

In [ ]:
Img(f'ml/reports/{sel}/misclassified.jpg')